# ADADelta

In [1]:
import nnfs
from nnfs.datasets import spiral_data
import numpy as np
import matplotlib.pyplot as plt
nnfs.init()

In [ ]:
class Layer_Dense:
    def __init__(self, n_inputs, n_neurons):
        self.weights = 0.01*np.random.randn(n_inputs, n_neurons)
        self.biases = np.zeros((1,n_neurons))
    def forward(self, X):
        self.inputs = X
        self.outputs = np.dot(X, self.weights) + self.biases
    def backward(self, dvalues):
        #dvalues is the gradient of the loss wrt the output of the layer
        self.dweights = np.dot(self.inputs.T, dvalues)
        self.dbiases = np.sum(dvalues, axis = 0, keepdims=True)
        self.dinputs = np.dot(dvalues, self.weights.T)
        
class Activation_ReLU:
    def forward(self, inputs):
        self.inputs = inputs
        self.outputs = np.maximum(0, inputs)
    def backward(self, dvalues):
        #First copy and then check <0 or >0
        self.dinputs = dvalues.copy()
        self.dinputs[self.inputs <= 0] = 0 
        
class Activation_Softmax:
    def forward(self, X):
        not_normalized = np.exp(X - np.max(X, axis = 1, keepdims=True))
        self.outputs = not_normalized/np.sum(not_normalized, axis = 1, keepdims=True)
        
class Loss:
    def calculate(self, y_pred, y_true):
        return np.mean(self.forward(y_pred, y_true))
    
class CategoricalCrossEntropy(Loss): #CCE inherits Loss
    def forward(self, y_pred, y_true):
        #Clip y_pred
        y_pred_clip = np.clip(y_pred, 1e-7, 1 - 1e-7)
        if len(y_true.shape) == 1: #Case 1: Sparse format [0,1,1,2]
            return -np.log(y_pred_clip[range(len(y_pred_clip)), y_true])
        elif len(y_true.shape) == 2: #Case 2: One Hot Encoding of Output
            return -np.log(np.sum(y_pred_clip*y_true, axis = 1)) #Element-wise multiplication
        #Note you are returning negative log likelihood matrix
    def backward(self, dvalues, y_true):
        samples = len(dvalues)
        labels = len(dvalues[0])
    
        #If class labels are sparse, convert to one hot encoded
        if len(y_true.shape) == 1:
            y_true = np.eye(labels)[y_true]
        
        self.dinputs = - y_true/dvalues         #Gradient Calculation
        self.dinputs = self.dinputs/samples     #Normalized with number of samples
        
class Activation_Softmax_CategoricalCrossEntropy:
    def __init__(self):
        self.loss = CategoricalCrossEntropy()
        self.activation = Activation_Softmax()
        
    def forward(self, inputs, y_true):
        self.activation.forward(inputs)
        self.outputs = self.activation.outputs
        return self.loss.calculate(self.outputs, y_true)
    
    def backward(self, dvalues, y_true):
        samples = len(dvalues)
        
        #Convert one-hot encoded y_true to sparse form
        if len(y_true.shape) == 2:
            y_true = np.argmax(y_true, axis = 1)
        
        self.dinputs = dvalues.copy()
        self.dinputs[range(samples), y_true] -= 1
        
        #Normalize this
        self.dinputs = self.dinputs/samples
    
class ADADelta:
    def __init__(self, rho = 0.99, epsilon = 1e-7):
        self.rho = rho
        self.epsilon = epsilon
        
    def update_params(self, layer):
        if not hasattr(layer, "weight_cache"):
            layer.weight_cache = np.zeros_like(layer.weights)
            layer.bias_cache = np.zeros_like(layer.biases)
        
        if not hasattr(layer, 'weight_updates'):
            layer.weight_updates = np.zeros_like(layer.weights)
            layer.bias_updates = np.zeros_like(layer.biases)
        
        #accumulating squared parameter gradients
        layer.weight_cache = self.rho*layer.weight_cache + (1.0 - self.rho)*(layer.dweights**2)
        layer.bias_cache = self.rho*layer.bias_cache + (1.0 - self.rho)*(layer.dbiases**2)
        
        #Calculating current parameter update based on past parameter update
        w_update = - (np.sqrt(layer.weight_updates + self.epsilon)/np.sqrt(layer.weight_cache + self.epsilon))*layer.dweights
        b_update = - (np.sqrt(layer.bias_updates + self.epsilon)/np.sqrt(layer.bias_cache + self.epsilon))*layer.dbiases
        
        #Updating parameters
        layer.weights += w_update
        layer.biases += b_update
        
        #Accumulating all the squared parameter updates
        layer.weight_updates = self.rho*(layer.weight_updates**2) + (1.0 - self.rho)*(w_update**2) 
        layer.bias_updates = self.rho*(layer.bias_updates**2) + (1.0 - self.rho)*(b_update**2)

In [13]:
X, y = spiral_data(samples = 100, classes = 3)

dense1 = Layer_Dense(2, 64)
activation1 = Activation_ReLU()
dense2 = Layer_Dense(64, 3)
loss_activation = Activation_Softmax_CategoricalCrossEntropy()
optimizer = ADADelta()

for epoch in range(10000):
    dense1.forward(X)
    activation1.forward(dense1.outputs)
    dense2.forward(activation1.outputs)
    loss = loss_activation.forward(dense2.outputs, y)
    y_pred = np.argmax(loss_activation.outputs, axis = 1)
    
    if len(y.shape) == 2:
        y_true = np.argmax(y, axis = 1)
    else:
        y_true = y
    
    accuracy = np.mean(y_true == y_pred)
    
    if (epoch+1)% 100 == 0:
        print(f"Epoch: {epoch+1}\tAccuracy: {accuracy:.3f}\tLoss:{loss:.3f}")
    
    loss_activation.backward(loss_activation.outputs,y_true)
    dense2.backward(loss_activation.dinputs)
    activation1.backward(dense2.dinputs)
    dense1.backward(activation1.dinputs)

    optimizer.update_params(dense1)
    optimizer.update_params(dense2)


Epoch: 100	Accuracy: 0.407	Loss:1.095
Epoch: 200	Accuracy: 0.410	Loss:1.087
Epoch: 300	Accuracy: 0.407	Loss:1.081
Epoch: 400	Accuracy: 0.410	Loss:1.078
Epoch: 500	Accuracy: 0.407	Loss:1.076
Epoch: 600	Accuracy: 0.417	Loss:1.075
Epoch: 700	Accuracy: 0.413	Loss:1.073
Epoch: 800	Accuracy: 0.410	Loss:1.071
Epoch: 900	Accuracy: 0.410	Loss:1.069
Epoch: 1000	Accuracy: 0.427	Loss:1.066
Epoch: 1100	Accuracy: 0.423	Loss:1.063
Epoch: 1200	Accuracy: 0.430	Loss:1.059
Epoch: 1300	Accuracy: 0.423	Loss:1.056
Epoch: 1400	Accuracy: 0.417	Loss:1.052
Epoch: 1500	Accuracy: 0.427	Loss:1.048
Epoch: 1600	Accuracy: 0.423	Loss:1.044
Epoch: 1700	Accuracy: 0.440	Loss:1.040
Epoch: 1800	Accuracy: 0.447	Loss:1.036
Epoch: 1900	Accuracy: 0.450	Loss:1.031
Epoch: 2000	Accuracy: 0.463	Loss:1.027
Epoch: 2100	Accuracy: 0.467	Loss:1.023
Epoch: 2200	Accuracy: 0.487	Loss:1.018
Epoch: 2300	Accuracy: 0.503	Loss:1.014
Epoch: 2400	Accuracy: 0.503	Loss:1.010
Epoch: 2500	Accuracy: 0.500	Loss:1.006
Epoch: 2600	Accuracy: 0.507	Loss:1